**Bridge steering save format (for multi_hop, prompt_engineering, few_shot):**  
If you use a separate bridge_steering notebook/script, save each method as a CSV in the bridge run folder (or `eval_outputs/`) with the **same columns** as `steering_baseline.csv`: `eval_idx`, `id`, `method` (or `tag`), `user_prompt`, `reply`, `reply_len_words`, `has_mechanism_line`, `ground_truth_B`, `ground_truth_C`, `problem_text`. Name files e.g. `steering_multi_hop.csv`, `steering_prompt_engineering.csv`, `steering_few_shot.csv`. This notebook will load them and include them in judge statistics by method.

**Why do "baseline" and "steering_baseline" get different judge scores?**

- **baseline**: From the SFT run (`baseline_eval_results.json`). Prompt is *"Alternative Uses Task … Give 8 unconventional uses. List them clearly."* with **no format or length rule**. The base model often produces **long prose** (numbered items or paragraphs).
- **steering_baseline**: From the bridge steering run (`steering_baseline.csv`). Prompt uses *"Object: X"* and **explicit rules**: *"One line per use, starting with '-'"*, *"Each use must be <= 12 words."* So replies are **short bullet lists**.

Different prompts → different reply style → different judge scores (e.g. longer SFT baseline can score higher on novelty; short steering bullets can be judged as more generic). To compare methods fairly, use one pipeline (either SFT-only or steering-only) or align prompts so both baselines use the same instruction and format.

# Evaluation-only: Bridge internalization (baseline + methods)

Runs all configured methods (baseline + optional SFT checkpoints) on the same eval set, saves DataFrames, and exports a **judge-friendly table** (problem + method → result) for LLM-as-judge.

**Config:** Set `TASK_TYPE`, `EVAL_SEED`, `N_EVAL`, `MODEL_NAME`, and `METHODS` (list of method name + optional checkpoint path).

In [ ]:
# DLP package (run from DLP/ or notebooks/)
import sys
from pathlib import Path
_root = Path.cwd()
for _ in range(10):
    if (_root / "dlp").is_dir():
        sys.path.insert(0, str(_root))
        break
    _root = _root.parent
from dlp.models import HFLoader
QwenModelLoader = HFLoader  # alias for rest of notebook


In [1]:
"""Config: task, data, model, and methods to evaluate.

Set these before running:
- TASK_TYPE: TaskType.AUT or TaskType.PS
- N_EVAL, EVAL_SEED: eval set size and seed
- MODEL_NAME: base model for generation (if not USE_PRELOADED)
- METHODS: list of (display_name, checkpoint_path); None = baseline
- SFT_RUN_DIR, BRIDGE_RUN_DIR: paths to pre-saved SFT and bridge steering outputs (or None)
- USE_PRELOADED: True to skip generation and load from CSVs
- JUDGE_MODEL: Prometheus model (see judge cell for JUDGE_BATCH_SIZE, JUDGE_AUT_BY_USE)
"""

import os
from pathlib import Path
from enum import Enum
class TaskType(Enum):
    AUT = "aut"
    PS = "ps"
DATASET_PATHS = {
    TaskType.PS: "dataset/abcd_aiden.json",
    TaskType.AUT: "dataset/abcd_aut.json",
}
TASK_TYPE = TaskType.AUT  # or TaskType.PS
EVAL_SEED = 7
N_EVAL = 10
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

# Methods: (display_name, checkpoint_path). None = baseline (base model, creative-only prompt).
# Add SFT checkpoints to compare, e.g. ("postsft", "sft_bridge_outputs/checkpoints/checkpoint-XXX").
METHODS = [
    ("baseline", None),
    ("postsft", "sft_bridge_outputs/checkpoints"),  # or path to specific checkpoint-XXX
]

EVAL_SYSTEM = "You are a helpful assistant."
MAX_NEW_TOKENS = 384
EVAL_OUTPUT_DIR = Path("eval_outputs")
EVAL_OUTPUT_DIR.mkdir(exist_ok=True)

# --- Output folders for LLM-as-judge (use pre-saved results) ---
# SFT run folder from sft_bridge_internalization (contains baseline_eval_results.json, postsft_eval_results.json).
SFT_RUN_DIR = Path("sft_bridge_outputs/aut_20260222_104905")  # e.g. "sft_bridge_outputs/aut_20260221_124750"
# Bridge steering run folder (timestamped). "latest" = use most recent in ../environment_uv; or set explicit path.
BRIDGE_RUN_DIR = Path("bridge_steering_outputs_aut_20260222_105439")  # or "../environment_uv/bridge_steering_outputs_aut_20260221_125537", or "latest", or None

# LLM-as-judge: Prometheus 2 (HF)
JUDGE_MODEL = "prometheus-eval/prometheus-7b-v2.0"  # or prometheus-eval/prometheus-8x7b-v2.0

# Use pre-saved results from bridge_steering and sft_bridge_internalization (no generation here)
USE_PRELOADED = True
# Resolve BRIDGE_RUN_DIR and build PRELOAD_PATHS (SFT from SFT_RUN_DIR or eval_outputs; steering from bridge run folder)
# Bridge steering should save CSVs with columns: eval_idx, id, method (or tag), user_prompt, reply, reply_len_words,
# has_mechanism_line, ground_truth_B, ground_truth_C, problem_text. Same schema as steering_alpha_sweep / steering_baseline.
_bridge_dir = None
if BRIDGE_RUN_DIR is not None:
    if str(BRIDGE_RUN_DIR) == "latest":
        _env = Path("../environment_uv")
        if _env.exists():
            _candidates = sorted(_env.glob(f"bridge_steering_outputs_{TASK_TYPE.value}_*"), key=lambda p: p.stat().st_mtime)
            if _candidates:
                _bridge_dir = _candidates[-1]
                print(f"Bridge run (latest): {_bridge_dir}")
    else:
        _bridge_dir = Path(BRIDGE_RUN_DIR)
        if not _bridge_dir.is_absolute() and not _bridge_dir.exists():
            _bridge_dir = Path("../environment_uv") / _bridge_dir
        if _bridge_dir.exists():
            print(f"Bridge run: {_bridge_dir}")
PRELOAD_PATHS = []
if SFT_RUN_DIR is None or not Path(SFT_RUN_DIR).exists():
    PRELOAD_PATHS.extend([EVAL_OUTPUT_DIR / "sft_baseline.csv", EVAL_OUTPUT_DIR / "sft_postsft.csv"])
# Bridge-run CSVs: alpha sweep, baseline, and optional multi_hop / prompt_engineering / few_shot (same schema)
BRIDGE_CSV_NAMES = [
    "steering_alpha_sweep.csv",
    "steering_baseline.csv",
    "steering_multi_hop.csv",
    "steering_prompt_engineering.csv",
    "steering_few_shot.csv",
]
if _bridge_dir is not None:
    for name in BRIDGE_CSV_NAMES:
        p = _bridge_dir / name
        if p.exists():
            PRELOAD_PATHS.append(p)
if _bridge_dir is None:
    PRELOAD_PATHS.append(EVAL_OUTPUT_DIR / "steering_alpha_sweep.csv")
    for name in ["steering_multi_hop.csv", "steering_prompt_engineering.csv", "steering_few_shot.csv"]:
        p = EVAL_OUTPUT_DIR / name
        if p.exists():
            PRELOAD_PATHS.append(p)


print(f"TASK_TYPE={TASK_TYPE.value}, N_EVAL={N_EVAL}, EVAL_SEED={EVAL_SEED}")
print(f"METHODS: {[m[0] for m in METHODS]}")
print(f"USE_PRELOADED: {USE_PRELOADED}")
print(f"Judge: Prometheus ({JUDGE_MODEL})")

Bridge run: ../environment_uv/bridge_steering_outputs_aut_20260222_105439
TASK_TYPE=aut, N_EVAL=10, EVAL_SEED=7
METHODS: ['baseline', 'postsft']
USE_PRELOADED: True
Judge: Prometheus (prometheus-eval/prometheus-7b-v2.0)


In [11]:
"""Imports and QwenModelLoader (same interface as SFT notebook)."""

from __future__ import annotations

import json
import os
import re
import subprocess
import sys
from pathlib import Path
from typing import Any

os.environ.setdefault("BNB_CUDA_VERSION", "121")
import tempfile as _tempfile
_fake_cuda = os.path.join(_tempfile.gettempdir(), "fake_cuda_nvcc_stub")
_fake_bin = os.path.join(_fake_cuda, "bin")
os.makedirs(_fake_bin, exist_ok=True)
_nvcc = os.path.join(_fake_bin, "nvcc")
with open(_nvcc, "w") as _f:
    _f.write('#!/bin/sh\n')
    _f.write('V="Cuda compilation tools, release 12.1, V12.1.0"\n')
    _f.write('case "$1" in -V) echo "$V" ;; --version) echo "$V" ;; esac\n')
    _f.write('exit 0\n')
os.chmod(_nvcc, 0o755)
os.environ["CUDA_HOME"] = _fake_cuda

import pandas as pd
import torch
from tqdm.auto import tqdm
# Force-load GenerationMixin before AutoModelForCausalLM (workaround for lazy-module ImportError in some envs)
from transformers.generation.utils import GenerationMixin  # noqa: F401
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    import peft  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "peft>=0.10.0"])


# QwenModelLoader from dlp.models (see previous cell)
print("Imports OK.")


Imports OK.


In [20]:
"""Load eval data and define prompt formatters (PS vs AUT, baseline vs full)."""

import random as _rng

DATASET_PATH = DATASET_PATHS[TASK_TYPE]
with open(DATASET_PATH) as f:
    abcd_data = json.load(f)

_eval_idxs = set(_rng.Random(EVAL_SEED).sample(range(len(abcd_data)), min(N_EVAL, len(abcd_data))))
eval_items = [item for i, item in enumerate(abcd_data) if i in _eval_idxs]


def format_user_prompt_ps(item: dict) -> str:
    problem = item["A"]
    return (
        f"Problem Solving Task\n\n"
        f"Problem: {problem}\n\n"
        f"Give one creative, non-obvious solution. State the mechanism in one line (Mechanism: <type> — <rule>), then your solution in a short paragraph."
    )


def format_user_prompt_baseline_ps(item: dict) -> str:
    problem = item["A"]
    return (
        f"Problem Solving Task\n\n"
        f"Problem: {problem}\n\n"
        f"Give one creative, non-obvious solution in a short paragraph."
    )


def format_user_prompt_aut(item: dict) -> str:
    task = item["A"]
    return (
        f"Alternative Uses Task\n\n"
        f"{task}\n\n"
        f"Give 8 unconventional uses. List them clearly."
    )


def format_user_prompt_baseline_aut(item: dict) -> str:
    return format_user_prompt_aut(item)


format_user_prompt = format_user_prompt_ps if TASK_TYPE.value == "ps" else format_user_prompt_aut
format_user_prompt_baseline = (
    format_user_prompt_baseline_ps if TASK_TYPE.value == "ps" else format_user_prompt_baseline_aut
)

print(f"Eval items: {len(eval_items)}, IDs: {[item['id'] for item in eval_items]}")

Eval items: 10, IDs: ['AUT_016_v2', 'AUT_018_v2', 'AUT_022_v2', 'AUT_028_v2', 'AUT_042_v2', 'AUT_022_v2_1', 'AUT_057_v2', 'AUT_065_v2', 'AUT_104_v2', 'AUT_116_v2']


In [21]:
"""Generate outputs for one method; return list of result dicts."""


def generate_eval_outputs(
    loader: QwenModelLoader,
    eval_items: list[dict],
    system_prompt: str | None,
    tag: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    user_prompt_fn=None,
) -> list[dict]:
    if user_prompt_fn is None:
        user_prompt_fn = format_user_prompt
    results = []
    for eval_idx, item in tqdm(enumerate(eval_items), total=len(eval_items), desc=f"Eval ({tag})"):
        user_prompt = user_prompt_fn(item)
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})
        reply = loader.generate(messages, max_new_tokens=max_new_tokens, do_sample=False)
        results.append({
            "eval_idx": eval_idx,
            "id": item["id"],
            "method": tag,
            "user_prompt": user_prompt,
            "reply": reply,
            "reply_len_words": len(reply.split()),
            "has_mechanism_line": bool(re.search(r"(?i)mechanism\s*:", reply)),
            "ground_truth_B": item["B"]["rule"],
            "ground_truth_C": item["C"],
        })
    return results

NameError: name 'QwenModelLoader' is not defined

In [6]:
import json
import pandas as pd 
"""Load pre-saved results from bridge_steering and sft_bridge_internalization (optional).
Bridge run CSVs (steering_alpha_sweep, steering_baseline, steering_multi_hop, steering_prompt_engineering,
steering_few_shot) must have columns: eval_idx, id, method (or tag), user_prompt, reply, ground_truth_B, ground_truth_C, etc."""
if USE_PRELOADED:
    loaded = []
    sft_dir = Path(SFT_RUN_DIR) if SFT_RUN_DIR else None
    if sft_dir is not None and sft_dir.exists():
        with open(DATASET_PATHS[TASK_TYPE]) as f:
            _full_data = json.load(f)
        id_to_A = {item["id"]: item["A"] for item in _full_data}
        for name, tag in [("baseline_eval_results.json", "baseline"), ("postsft_eval_results.json", "postsft")]:
            p = sft_dir / name
            if p.exists():
                with open(p) as f:
                    rows = json.load(f)
                for r in rows:
                    r["problem_text"] = id_to_A.get(r["id"], r.get("user_prompt", ""))
                    r["method"] = r.get("tag", tag)
                loaded.append(pd.DataFrame(rows))
        if loaded:
            print(f"SFT run: loaded from {sft_dir}")
    for p in PRELOAD_PATHS:
        if Path(p).exists():
            df = pd.read_csv(p)
            if "tag" in df.columns and "method" not in df.columns:
                df = df.rename(columns={"tag": "method"})
            loaded.append(df)
        else:
            print(f"Skip (not found): {p}")
    if loaded:
        combined_df = pd.concat(loaded, ignore_index=True)
        # Build eval_items from first occurrence per eval_idx (for judge table)
        first_per_idx = combined_df.drop_duplicates("eval_idx").sort_values("eval_idx")
        eval_items = [
            {"id": r["id"], "A": r["problem_text"], "B": {"rule": r["ground_truth_B"]}, "C": r["ground_truth_C"]}
            for _, r in first_per_idx.iterrows()
        ]
        print(f"Preloaded: {len(combined_df)} rows, methods: {combined_df['method'].unique().tolist()}, eval_items: {len(eval_items)}")
    else:
        combined_df = None
        print("No preloaded files found; run the next cell to generate.")
else:
    combined_df = None

SFT run: loaded from sft_bridge_outputs/aut_20260222_104905
Preloaded: 130 rows, methods: ['baseline', 'postsft', 'steering_alpha_0.0', 'steering_alpha_0.25', 'steering_alpha_0.5', 'steering_alpha_0.75', 'steering_alpha_1.0', 'steering_alpha_1.25', 'steering_alpha_1.5', 'steering_baseline', 'steering_multi_hop', 'steering_prompt_engineering', 'steering_few_shot'], eval_items: 10


In [22]:
"""Run all methods: baseline (base model) and each SFT checkpoint. Skipped if USE_PRELOADED and preload succeeded.
When using preloaded results, do not run this cell so the judge (next sections) can use the GPU."""
if not USE_PRELOADED or combined_df is None:
    from peft import PeftModel

    loader = QwenModelLoader(MODEL_NAME)
    loader.load()

    all_results: list[dict] = []

    for method_name, checkpoint_path in METHODS:
        is_baseline = checkpoint_path is None
        user_prompt_fn = format_user_prompt_baseline if is_baseline else format_user_prompt

        if is_baseline:
            results = generate_eval_outputs(
                loader, eval_items, EVAL_SYSTEM, tag=method_name,
                user_prompt_fn=user_prompt_fn,
            )
        else:
            ckpt = Path(checkpoint_path)
            if not ckpt.exists():
                print(f"Skipping {method_name}: path not found {ckpt}")
                continue
            if ckpt.is_dir():
                subdirs = sorted([d for d in ckpt.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")])
                if not subdirs:
                    print(f"Skipping {method_name}: no checkpoint-* in {ckpt}")
                    continue
                ckpt = subdirs[-1]
            peft_model = PeftModel.from_pretrained(loader.model, str(ckpt))
            loader._model = peft_model
            peft_model.eval()
            results = generate_eval_outputs(
                loader, eval_items, EVAL_SYSTEM, tag=method_name,
                user_prompt_fn=user_prompt_fn,
            )
            loader._model = peft_model.unload()

        all_results.extend(results)

    combined_df = pd.DataFrame(all_results)
    print(f"Total rows: {len(combined_df)}, methods: {combined_df['method'].unique().tolist()}")
else:
    print("Using preloaded results (combined_df and eval_items from previous cell).")

Using preloaded results (combined_df and eval_items from previous cell).


In [23]:
"""Save DataFrames: combined long-format + per-method."""

def _to_str(x):
    if isinstance(x, list):
        return " | ".join(str(v) for v in x)
    return x if isinstance(x, str) else ("" if pd.isna(x) else str(x))

# Normalize columns that can be list or str (parquet cannot mix types)
_df_parquet = combined_df.copy()
for col in _df_parquet.columns:
    if _df_parquet[col].dtype == object:
        _df_parquet[col] = _df_parquet[col].apply(_to_str)

combined_path = EVAL_OUTPUT_DIR / "eval_combined.parquet"
try:
    _df_parquet.to_parquet(combined_path, index=False)
    print(f"Saved: {combined_path}")
except Exception as e:
    print(f"Parquet save skipped ({e}). Saving CSV only.")

combined_csv = EVAL_OUTPUT_DIR / "eval_combined.csv"
combined_df.to_csv(combined_csv, index=False)
print(f"Saved: {combined_csv}")

for method in combined_df["method"].unique():
    sub = combined_df[combined_df["method"] == method]
    path = EVAL_OUTPUT_DIR / f"eval_{method}.csv"
    sub.to_csv(path, index=False)
    print(f"Saved: {path}")

display(combined_df.head())

Saved: eval_outputs/eval_combined.parquet
Saved: eval_outputs/eval_combined.csv
Saved: eval_outputs/eval_baseline.csv
Saved: eval_outputs/eval_postsft.csv
Saved: eval_outputs/eval_steering_alpha_0.0.csv
Saved: eval_outputs/eval_steering_alpha_0.25.csv
Saved: eval_outputs/eval_steering_alpha_0.5.csv
Saved: eval_outputs/eval_steering_alpha_0.75.csv
Saved: eval_outputs/eval_steering_alpha_1.0.csv
Saved: eval_outputs/eval_steering_alpha_1.25.csv
Saved: eval_outputs/eval_steering_alpha_1.5.csv
Saved: eval_outputs/eval_steering_baseline.csv
Saved: eval_outputs/eval_steering_multi_hop.csv
Saved: eval_outputs/eval_steering_prompt_engineering.csv
Saved: eval_outputs/eval_steering_few_shot.csv


,eval_idx,id,tag,system,user_prompt,reply,reply_len_words,has_mechanism_line,ground_truth_B,ground_truth_C,problem_text,method
0,0,AUT_016_v2,baseline,You are a helpful assistant.,Alternative Uses Task\n\nGive 8 unconventional...,1. **Polishing Precious Metals**: Sandpaper ca...,248,False,Treat it as a controllable abrasive transfer f...,[Friction pad to prevent a clamp from slipping...,Give 8 unconventional uses for sandpaper focus...,baseline
1,1,AUT_018_v2,baseline,You are a helpful assistant.,Alternative Uses Task\n\nGive 8 unconventional...,1. **Thermal Conductivity Tester**: Use the me...,311,False,Treat it as a metal thermal conductor + reflec...,[Heat sink handle: clamp spoon to a small warm...,Give 8 unconventional uses for a metal spoon f...,baseline
2,2,AUT_022_v2,baseline,You are a helpful assistant.,Alternative Uses Task\n\nGive 8 unconventional...,1. **Mechanical Fastener**: In mechanical desi...,309,False,Treat it as a spring clamp with removable hand...,[Quick clamp to hold a glued joint while curin...,Give 8 unconventional uses for a binder clip f...,baseline
3,3,AUT_028_v2,baseline,You are a helpful assistant.,Alternative Uses Task\n\nGive 8 unconventional...,1. **User Interface Wireframing:** Use sticky ...,310,False,Treat each note as a movable state token in a ...,[Throughput map: move one note per step to rev...,Give 8 unconventional uses for sticky notes fo...,baseline
4,4,AUT_042_v2,baseline,You are a helpful assistant.,Alternative Uses Task\n\nGive 8 unconventional...,1. **Shear Stress Reducer**: In the field of f...,315,False,Treat it as a high-surface-area fibrous bounda...,[Anti-slip mat: increase static friction under...,Give 8 unconventional uses for a towel focusin...,baseline


In [24]:
"""Build judge-friendly table: one row per (eval item × method) with problem_id, method, reply, problem_text."""

# Minimal columns for judge: problem_id, method, reply, problem_text (no gold/reference)
judge_long = combined_df[["id", "method", "reply"]].copy()
judge_long = judge_long.rename(columns={"id": "problem_id"})

# Add problem text (same for all methods per id)
id_to_A = {item["id"]: item["A"] for item in eval_items}
judge_long["problem_text"] = judge_long["problem_id"].map(id_to_A)

judge_long_path = EVAL_OUTPUT_DIR / "judge_table_long.csv"
judge_long.to_csv(judge_long_path, index=False)
print(f"Judge table (long): {judge_long_path}")
display(judge_long.head(4))

Judge table (long): eval_outputs/judge_table_long.csv


,problem_id,method,reply,problem_text
0,AUT_016_v2,baseline,1. **Polishing Precious Metals**: Sandpaper ca...,Give 8 unconventional uses for sandpaper focus...
1,AUT_018_v2,baseline,1. **Thermal Conductivity Tester**: Use the me...,Give 8 unconventional uses for a metal spoon f...
2,AUT_022_v2,baseline,1. **Mechanical Fastener**: In mechanical desi...,Give 8 unconventional uses for a binder clip f...
3,AUT_028_v2,baseline,1. **User Interface Wireframing:** Use sticky ...,Give 8 unconventional uses for sticky notes fo...


In [43]:
"""LLM judge: Prometheus 2 (HF). Setup + prompt builders + parsing."""

import ast
import json
import re

# --- Prometheus 2 (Absolute Grading format) — criteria for both AUT and PS ---
PROMETHEUS_SYSTEM = "You are a fair judge assistant tasked with providing clear, objective feedback based on specific criteria, ensuring each assessment reflects the absolute standards set for performance."

# PS: creative problem-solving (novelty, usability)
PROMETHEUS_CRITERIA_PS = "Quality of the creative problem-solving reply: novelty and usability."
PROMETHEUS_SCORE_DESCRIPTIONS_PS = """Score 1: Conventional; not usable.
Score 2: Below average; low novelty; limited usability.
Score 3: Adequate; moderate novelty; somewhat usable.
Score 4: Good; novel elements; practical usability.
Score 5: Excellent; highly novel; directly applicable."""

# AUT: alternative uses (clarity/variety of uses, originality, practicality)
PROMETHEUS_CRITERIA_AUT = "Quality of the alternative-uses reply: clarity and variety of uses, originality, and practicality/usability."
PROMETHEUS_SCORE_DESCRIPTIONS_AUT = """Score 1: Few or no clear uses; unoriginal; repetitive; not usable.
Score 2: Few uses; weak originality; limited variety; limited usability.
Score 3: Reasonable number of uses; moderate originality; some variety; somewhat usable.
Score 4: Good number of clear uses; novel elements; good variety; practical.
Score 5: Many clear, original uses; high variety; highly practical and applicable."""

# Select by task (used in build_prometheus_instruction)
if TASK_TYPE.value == "ps":
    PROMETHEUS_CRITERIA = PROMETHEUS_CRITERIA_PS
    PROMETHEUS_SCORE_DESCRIPTIONS = PROMETHEUS_SCORE_DESCRIPTIONS_PS
else:
    PROMETHEUS_CRITERIA = PROMETHEUS_CRITERIA_AUT
    PROMETHEUS_SCORE_DESCRIPTIONS = PROMETHEUS_SCORE_DESCRIPTIONS_AUT

# AUT-only: judge each use separately ("per_use") or one JSON with all uses ("json"). None = holistic prompt.
JUDGE_AUT_BY_USE = "per_use" # "per_use" | "json" | None

def _extract_object_from_problem(problem_text: str) -> str:
    """Extract object name from AUT prompt, e.g. 'sandpaper', 'a towel'."""
    m = re.search(r"(?:for|of)\s+(?:a\s+|an\s+)?([a-zA-Z\s]+?)(?:\s+focusing|\s+in\s+an|\s*$)", problem_text or "", re.IGNORECASE)
    return (m.group(1).strip() if m else "object").lower()

def split_reply_into_uses(reply, max_uses: int = 8) -> list[str]:
    """Split model reply into list of use strings. Handles: (1) already a list of strings,
    (2) string that looks like a Python list, (3) numbered/bullet lines (1) 2) or 1. 2. or - )."""
    print(f"[split_reply_into_uses] input reply len={len(reply) if isinstance(reply, str) else 'list-' + str(len(reply))}, max_uses={max_uses}")
    if reply is None:
        reply = ""
    # Already a list of strings (e.g. from JSON/parsed column)
    if isinstance(reply, list):
        parts = [str(x).strip() for x in reply if str(x).strip()][:max_uses]
        print(f"[split_reply_into_uses] reply was list -> {len(parts)} uses; first 100 chars: {repr((parts[0][:100] if parts else ''))}")
        return parts
    text = (reply or "").strip()
    if not text:
        print("[split_reply_into_uses] empty reply -> []")
        return []
    # String that looks like a Python list of quoted strings
    if text.startswith("[") and ("', " in text or '", ' in text):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                parts = [str(x).strip() for x in parsed if str(x).strip()][:max_uses]
                print(f"[split_reply_into_uses] parsed as list literal -> {len(parts)} uses; first 100 chars: {repr((parts[0][:100] if parts else ''))}")
                return parts
        except (ValueError, SyntaxError):
            pass
    # Try "1) ... 2) ..." or "1. ... 2. ..." or "- ... - ..."
    parts = re.split(r"\n\s*(?:\d+[.)]\s*|\-\s*)", text, maxsplit=max_uses)
    parts = [p.strip() for p in parts if p.strip()]
    # Drop leading intro line if it doesn't look like a use (e.g. "Here are eight...")
    if parts and len(parts[0]) > 80 and re.match(r"^(Here are|These are|Below|The following)", parts[0], re.I):
        parts = parts[1:]
    out = parts[:max_uses] if parts else [text[:500]]
    print(f"[split_reply_into_uses] -> {len(out)} uses; first 100 chars of first: {repr((out[0][:100] if out else ''))}")
    return out

def build_judge_per_use_instruction(object_name: str, use_text: str) -> str:
    """Structured per-use evaluation: novelty + usability rubric and JSON output format."""
    use_preview = (use_text or "")[:500]
#     return """
#     ou are a strict evaluator. Output ONLY one valid JSON object (no preamble, no markdown, no extra keys, no trailing text).

# Object: {object_name}
# Proposed use: {use_preview}

# Score TWO metrics (integers 1–10). Then compute an overall score as the rounded mean:
# overall = round((novelty + usability) / 2)

# 1) Novelty (1–10)
# How uncommon and non-obvious this use is for this object in this setting.
# Score the IDEA only — do NOT reward verbosity, style, or adding unrelated components.
# Guidance:
# - 1–2: very common / first-thought / cliché for this object
# - 3–4: common pattern with a small twist
# - 5–6: clearly different from typical answers, but not rare
# - 7–8: rare and surprising yet still coherent
# - 9–10: extremely rare; would surprise most evaluators

# Caps:
# - Common trope / standard advice ⇒ novelty ≤ 4
# - Generic “add AI/sensors/solar/etc.” with no object-specific idea ⇒ novelty ≤ 3
# - The object is interchangeable (same idea works with many objects) ⇒ novelty ≤ 4

# 2) Usability (1–10)
# How likely this would work as stated using realistic constraints and typical resources.
# Penalize missing steps, hidden requirements, contradictions, and fragile assumptions.
# Guidance:
# - 1–2: effectively won’t work / contradicts reality or constraints
# - 3–4: barely plausible; requires major missing resources or perfect conditions
# - 5–6: plausible but underspecified or inefficient; needs extra assumptions/effort
# - 7–8: practical and executable with modest effort; works in typical conditions
# - 9–10: highly reliable, low-friction, and robust

# Caps:
# - Requires major unspecified resources / unrealistic conditions ⇒ usability ≤ 4
# - Physically implausible / contradictory ⇒ usability ≤ 2
# - Unsafe/damaging unless mitigation is stated ⇒ usability ≤ 5

# Return JSON exactly:
# {
#   "analysis": {
#     "key_object_property": "string",
#     "reasoning": "string (<= 20 words)"
#   },
#   "scores": {
#     "novelty": integer,
#     "usability": integer,
#     "overall": integer
#   }
# }
#     """
    return f"""<s> [INST] You are a highly critical and fair evaluator. Your task is to analyze a specific use-case for an object and output your evaluation as a RAW JSON object only. No preamble, no markdown formatting.

### Task
Evaluate the following creative use-case for an object based on two distinct metrics: Novelty and Usability.

### Input Data
- Object: {object_name}
- Proposed Use: {use_preview}

### Evaluation Rubric
1) Novelty (Range: 1–10)
Novelty = statistical rarity + non-obviousness of the core mechanism for this object+prompt.
Do NOT reward verbosity, fancy wording, or extra components. Score the IDEA.

Percentile anchors (calibrate against typical responses for this object):
- 1–2: Bottom 20% (default/cliché; “first-thought” ideas; very common templates)
- 3–4: 20–50% (common with small twist; familiar pattern; minor recombination)
- 5–6: 50–80% (clearly different from default set; concrete mechanism, but not rare)
- 7–8: 80–95% (rare; exploits a specific constraint/property in a non-obvious way)
- 9: Top 5% (extremely rare; crisp, high-leverage mechanism that surprises most evaluators)
- 10: Top ~1% (breakthrough-level leap; almost never; still must be coherent and physical)

Hard rules:
- If it resembles common DIY/Pinterest/standard advice, Novelty ≤ 4.
- If it’s a generic “wishlist” (add sensors/AI/renewables/etc.) without a distinct mechanism, Novelty ≤ 3.
- If unsure between two scores, choose the LOWER score.
- In a batch, 9–10 should be rare; 1–3 should appear sometimes.

2) **Usability (Range: 1-10)**
Score feasibility and likelihood of working under stated constraints (coherent, implementable, robust).
Penalize contradictions, missing requirements, and fragile assumptions.

Anchors:
- **1–2**: Impossible/contradictory; violates constraints or fails immediately.
- **3–4**: Barely plausible; needs major missing resources or very specific conditions.
- **5–6**: Plausible but underspecified/inefficient; works with significant effort or extra assumptions.
- **7–8**: Practical and coherent; executable with modest effort; works in typical conditions.
- **9**: Very reliable and low-friction within constraints; handles common edge cases.
- **10**: Near-certain success; elegant, robust, and scalable within constraints.

### Instructions
1. Identify the specific **Functional Property** being utilized (e.g., abrasive friction, high-surface-area texture, etc.).
2. Provide a brief **Reasoning** (max 20 words) for the scores assigned.
3. OUTPUT ONLY THE JSON. Do not include markdown code blocks.

### JSON Output Format
{{
  "analysis": {{
    "functional_property": "string",
    "reasoning": "string"
  }},
  "scores": {{
    "novelty": integer,
    "usability": integer
  }}
}} [/INST]"""
def parse_judge_per_use_response(text: str) -> dict:
    """Parse per-use response: strip [/INST] echo; extract novelty,usability (CSV or 'N, M'); overall from [RESULT] N or prose 'score is N'."""
    out = {"novelty": None, "usability": None, "overall_score": None, "reasoning": ""}
    text = (text or "").strip()
    if "[/INST]" in text:
        text = text.split("[/INST]", 1)[-1].strip()
    # Find each score separately: "novelty": 4, "usability": 5
    m = re.search(r'"novelty"\s*:\s*(\d+)', text)
    if m:
        out["novelty"] = int(m.group(1))
    m = re.search(r'"usability"\s*:\s*(\d+)', text)
    if m:
        out["usability"] = int(m.group(1))
    # Fallback: "N, M" in text (N,M 1-5)
    if out["novelty"] is None or out["usability"] is None:
        two_nums = re.search(r"\b([1-5])\s*,\s*([1-5])\b", text)
        if two_nums:
            if out["novelty"] is None:
                out["novelty"] = int(two_nums.group(1))
            if out["usability"] is None:
                out["usability"] = int(two_nums.group(2))
    # Prose: "novelty (2)", "2 for novelty", "usability is 4", etc.
    if out["novelty"] is None:
        for pat in [r"(\d)\s+for\s+novelty", r"\bnovelty\s*[:\-(]?\s*(\d)\b", r"\bnovelty\s+is\s+(\d)\b", r"\bnovelty\s+score\s+(?:is|of)\s+(\d)\b", r"\bnovelty\s+of\s+(\d)\b"]:
            m = re.search(pat, text, re.I)
            if m:
                out["novelty"] = int(m.group(1))
                break
    if out["usability"] is None:
        for pat in [r"(\d)\s+for\s+usability", r"\busability\s*[:\-(]?\s*(\d)\b", r"\busability\s+is\s+(\d)\b", r"\busability\s+score\s+(?:is|of)\s+(\d)\b", r"\busability\s+of\s+(\d)\b"]:
            m = re.search(pat, text, re.I)
            if m:
                out["usability"] = int(m.group(1))
                break
    if out["novelty"] is not None and out["usability"] is not None:
        out["overall_score"] = round((out["novelty"] + out["usability"]) / 2)
    # Overall: [RESULT] N first, then prose fallbacks
    if out["overall_score"] is None:
        m = re.search(r"\[RESULT\]\s*(\d)", text, re.I)
        if m:
            out["overall_score"] = min(5, max(1, int(m.group(1))))
    if out["overall_score"] is None:
        for pat in [r"overall\s+score\s+is\s+(\d)", r"score\s+is\s+(\d)", r"the\s+score\s+is\s+(\d)", r"score\s+of\s+(\d)"]:
            m = re.search(pat, text, re.I)
            if m:
                out["overall_score"] = min(5, max(1, int(m.group(1))))
                break
    # If we have overall but only one of novelty/usability, infer the other so row has both
    if out["overall_score"] is not None:
        if out["novelty"] is not None and out["usability"] is None:
            out["usability"] = min(5, max(1, 2 * out["overall_score"] - out["novelty"]))
        elif out["novelty"] is None and out["usability"] is not None:
            out["novelty"] = min(5, max(1, 2 * out["overall_score"] - out["usability"]))
    out["reasoning"] = text[:200]
    return out

def build_judge_json_instruction(object_name: str, uses_list: list[str]) -> str:
    """Prometheus-aligned prompt: Absolute Grading structure (###Task, ###Instruction, ###Response, ###Score Rubrics, ###Feedback). Use with apply_chat_template (Mistral [INST]) per model card."""
    n = len(uses_list)
    uses_block = "\n".join(f"{i}) {u[:300]}" for i, u in enumerate(uses_list, 1))
    return f"""###Task Description:
You must output ONLY valid JSON. No other text, no markdown, no prose.
Return a JSON object with exactly these keys: "uses", "overall".
- "uses": array of {n} objects with keys: i, novelty, usability (i 1..{n}, each once; novelty/usability integers 1..5).
- "overall": object with keys: novelty, usability, result, reason (integers 1..5; result = round((novelty+usability)/2); reason <= 20 words).
The output must be parseable by json.loads.

###The instruction to evaluate:
Score each use of the object on novelty and usability. Object: {object_name}

###Response to evaluate:
{uses_block}

###Score Rubrics:
novelty: 1 obvious/common, 3 some twist, 5 surprising/uncommon
usability: 1 vague/unsafe, 3 somewhat actionable, 5 clear+realistic

###Feedback:"""

def _extract_brace_block(text: str, start: int) -> str | None:
    """From text[start] which should be '{', return the substring up to matching '}' (brace-count)."""
    if start >= len(text) or text[start] != "{":
        return None
    depth = 0
    i = start
    while i < len(text):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
        i += 1
    return None


def _extract_bracket_block(text: str, start: int) -> str | None:
    """From text[start] which should be '[', return the substring up to matching ']'."""
    if start >= len(text) or text[start] != "[":
        return None
    depth = 0
    i = start
    while i < len(text):
        if text[i] == "[":
            depth += 1
        elif text[i] == "]":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
        i += 1
    return None


def parse_judge_json_response(text: str) -> dict:
    """Parse JSON response with uses[] and overall.{novelty, usability}. Supports root object, root array+overall, and regex fallback. Extracts reasoning from post-JSON text or overall.reasoning."""
    out = {"overall_score": None, "novelty": None, "usability": None, "reasoning": "", "uses": []}
    raw_text = (text or "").strip()
    if "[/INST]" in raw_text:
        text = raw_text.split("[/INST]", 1)[-1].strip()
    else:
        text = raw_text
    print(f"[parse_judge_json_response] input len={len(text)}, first 200 chars: {repr(text[:200])}")

    def set_reasoning(tail: str):
        tail = (tail or "").strip()
        if tail and not tail.startswith("{"):
            out["reasoning"] = tail[:400].strip()

    # 1) Try root object {"uses": [...] or "uses": [...] — find the { that starts the root object
    for m in re.finditer(r'\{\s*"uses"\s*:\s*\[', text):
        snippet = _extract_brace_block(text, m.start())
        if snippet:
            print(f"[parse_judge_json_response] extracted root object (len={len(snippet)}): {snippet[:200]!r}...")
            try:
                d = json.loads(snippet)
                uses = d.get("uses") or []
                overall = d.get("overall") or {}
                if not overall and (d.get("novelty") is not None or d.get("usability") is not None or d.get("result") is not None):
                    overall = d
                out["novelty"] = overall.get("novelty")
                out["usability"] = overall.get("usability")
                if out["novelty"] is not None and isinstance(out["novelty"], float):
                    out["novelty"] = round(out["novelty"])
                if out["usability"] is not None and isinstance(out["usability"], float):
                    out["usability"] = round(out["usability"])
                out["uses"] = [{"i": u.get("i"), "novelty": u.get("novelty"), "usability": u.get("usability")} for u in uses]
                if out["novelty"] is None and uses:
                    nvs = [u.get("novelty") for u in uses if u.get("novelty") is not None]
                    if nvs:
                        out["novelty"] = round(sum(float(x) for x in nvs) / len(nvs))
                if out["usability"] is None and uses:
                    uvs = [u.get("usability") for u in uses if u.get("usability") is not None]
                    if uvs:
                        out["usability"] = round(sum(float(x) for x in uvs) / len(uvs))
                if out["novelty"] is not None and out["usability"] is not None:
                    out["overall_score"] = round((out["novelty"] + out["usability"]) / 2)
                reason_text = overall.get("reason") or overall.get("reasoning")
                if isinstance(reason_text, str) and reason_text.strip():
                    out["reasoning"] = reason_text.strip()[:400]
                else:
                    set_reasoning(text[m.start() + len(snippet) :])
                print(f"[parse_judge_json_response] parsed: uses={len(out['uses'])}, novelty={out['novelty']}, usability={out['usability']}, overall={out['overall_score']}")
                return out
            except json.JSONDecodeError:
                pass

    # 2) Judge often outputs [ {...}, ... ] then "overall": {...}. Parse array + overall separately.
    if text.strip().startswith("["):
        arr_match = _extract_bracket_block(text, text.index("["))
        if arr_match:
            try:
                uses = json.loads(arr_match)
                if isinstance(uses, list):
                    out["uses"] = [{"i": u.get("i"), "novelty": u.get("novelty"), "usability": u.get("usability")} for u in uses if isinstance(u, dict)]
                    rest = text[text.index("]") + 1 :]
                    overall_m = re.search(r'"overall"\s*:\s*\{', rest)
                    if overall_m:
                        overall_block = _extract_brace_block(rest, overall_m.end() - 1)
                        if overall_block:
                            try:
                                overall = json.loads(overall_block)
                                n, u = overall.get("novelty"), overall.get("usability")
                                if n is not None:
                                    out["novelty"] = round(float(n)) if isinstance(n, (int, float)) else n
                                if u is not None:
                                    out["usability"] = round(float(u)) if isinstance(u, (int, float)) else u
                                if out["novelty"] is None and out["uses"]:
                                    nvs = [x.get("novelty") for x in out["uses"] if x.get("novelty") is not None]
                                    if nvs:
                                        out["novelty"] = round(sum(float(x) for x in nvs) / len(nvs))
                                if out["usability"] is None and out["uses"]:
                                    uvs = [x.get("usability") for x in out["uses"] if x.get("usability") is not None]
                                    if uvs:
                                        out["usability"] = round(sum(float(x) for x in uvs) / len(uvs))
                                if out["novelty"] is not None and out["usability"] is not None:
                                    out["overall_score"] = round((out["novelty"] + out["usability"]) / 2)
                                elif overall.get("result") is not None:
                                    out["overall_score"] = min(5, max(1, round(float(overall["result"]))))
                                reason_text = overall.get("reason") or overall.get("reasoning")
                                if isinstance(reason_text, str):
                                    out["reasoning"] = reason_text.strip()[:400]
                            except json.JSONDecodeError:
                                pass
                    if not out["reasoning"]:
                        set_reasoning(rest)
                    if out["overall_score"] is not None or out["novelty"] is not None:
                        print(f"[parse_judge_json_response] parsed (array+overall): uses={len(out['uses'])}, novelty={out['novelty']}, usability={out['usability']}, overall={out['overall_score']}")
                        return out
            except json.JSONDecodeError:
                pass

    # 3) Greedy { ... } then regex overall fallback (allow floats)
    m = re.search(r"\{[\s\S]*\}", text)
    if m:
        snippet = m.group(0)
        print(f"[parse_judge_json_response] extracted JSON snippet (len={len(snippet)}): {snippet[:200]!r}...")
        try:
            d = json.loads(snippet)
            uses = d.get("uses") or []
            overall = d.get("overall") or {}
            if not overall and (d.get("novelty") is not None or d.get("usability") is not None or d.get("result") is not None):
                overall = d  # judge output only Overall: { novelty, usability, result, reasoning }
            out["novelty"] = overall.get("novelty")
            out["usability"] = overall.get("usability")
            if out["novelty"] is not None and isinstance(out["novelty"], float):
                out["novelty"] = round(out["novelty"])
            if out["usability"] is not None and isinstance(out["usability"], float):
                out["usability"] = round(out["usability"])
            out["uses"] = [{"i": u.get("i"), "novelty": u.get("novelty"), "usability": u.get("usability")} for u in uses]
            if out["novelty"] is None and uses:
                nvs = [u.get("novelty") for u in uses if u.get("novelty") is not None]
                if nvs:
                    out["novelty"] = round(sum(float(x) for x in nvs) / len(nvs))
            if out["usability"] is None and uses:
                uvs = [u.get("usability") for u in uses if u.get("usability") is not None]
                if uvs:
                    out["usability"] = round(sum(float(x) for x in uvs) / len(uvs))
            if out["novelty"] is not None and out["usability"] is not None:
                out["overall_score"] = round((out["novelty"] + out["usability"]) / 2)
            elif overall.get("result") is not None:
                out["overall_score"] = min(5, max(1, round(float(overall["result"]))))
            reason_text = overall.get("reason") or overall.get("reasoning")
            if isinstance(reason_text, str):
                out["reasoning"] = reason_text.strip()[:400]
            else:
                set_reasoning(text[m.start() + len(snippet) :])
            print(f"[parse_judge_json_response] parsed: uses={len(out['uses'])}, novelty={out['novelty']}, usability={out['usability']}, overall={out['overall_score']}")
            return out
        except json.JSONDecodeError as e:
            print(f"[parse_judge_json_response] JSONDecodeError: {e}; snippet: {snippet[:300]!r}")
            set_reasoning(text[m.start() + len(snippet) :])
    else:
        print("[parse_judge_json_response] no {...} found in response")

    # 4) Regex fallback for overall (allow floats: 3.5, 4.2)
    overall_m = re.search(r'"overall"\s*:\s*\{\s*"novelty"\s*:\s*([\d.]+)\s*,\s*"usability"\s*:\s*([\d.]+)\s*,\s*"result"\s*:\s*([\d.]+)\s*\}', text)
    if overall_m:
        try:
            out["novelty"] = round(float(overall_m.group(1)))
            out["usability"] = round(float(overall_m.group(2)))
            out["overall_score"] = min(5, max(1, round(float(overall_m.group(3)))))
            print(f"[parse_judge_json_response] recovered from overall: novelty={out['novelty']}, usability={out['usability']}, overall={out['overall_score']}")
        except ValueError:
            pass
    return out

def _run_judge_batch_raw(texts: list[str], max_new_tokens: int = 256):
    """Run judge on a batch of prompt texts (chat-templated for Prometheus/Mistral [INST] format); return list of decoded response strings."""
    if not texts or judge_model is None or judge_tokenizer is None:
        return [""] * len(texts)
    old_side = judge_tokenizer.padding_side
    judge_tokenizer.padding_side = "left"
    if judge_tokenizer.pad_token_id is None:
        judge_tokenizer.pad_token_id = judge_tokenizer.eos_token_id
    enc = judge_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=2048, return_attention_mask=True)
    enc = {k: v.to(judge_model.device) for k, v in enc.items()}
    input_lengths = enc["attention_mask"].sum(dim=1).tolist()
    try:
        out_ids = judge_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=judge_tokenizer.pad_token_id)
    finally:
        judge_tokenizer.padding_side = old_side
    return [judge_tokenizer.decode(out_ids[i, input_lengths[i]:], skip_special_tokens=True) for i in range(len(texts))]


def build_prometheus_instruction(row) -> str:
    """Build judge prompt: task + response to score + rubric. No reference — score the response on its own merits."""
    reply_preview = (row.get("reply") or "")[:200]
    print(f"[build_prometheus_instruction] reply len={len(row.get('reply') or '')} preview: {repr(reply_preview)}")
    instruction = (
        "###Task Description:\n"
        "Score the following response on its own merits. No reference answer. "
        "1. In the FIRST line state exactly: novelty (N), usability (N) with N 1-5 each. Example: novelty (4), usability (3). "
        "2. One sentence reasoning. "
        "3. End with [RESULT] followed by one integer 1-5 (the rounded mean of novelty and usability).\n\n"
        "###Instruction given to the model:\n" + row["problem_text"][:500] + "\n\n"
        "###Response to score:\n" + row["reply"][:1000] + "\n\n"
        "###Score Rubric:\n" + PROMETHEUS_CRITERIA + "\n" + PROMETHEUS_SCORE_DESCRIPTIONS + "\n\n"
        "###Feedback: "
    )
    return instruction

def parse_prometheus_response(text: str) -> dict:
    """Parse Prometheus output: 'Feedback: ... [RESULT] N' and optional sub-scores from feedback."""
    out = {"overall_score": None, "solution_quality": None, "novelty": None, "usability": None, "reasoning": ""}
    text = (text or "").strip()
    print(f"[parse_prometheus_response] input len={len(text)}, first 250 chars: {repr(text[:250])}")
    feedback = re.sub(r"\[RESULT\].*", "", text, flags=re.IGNORECASE).strip()
    feedback = re.sub(r"^###?Feedback:\s*", "", feedback, flags=re.IGNORECASE).strip()
    reasoning_lines = [ln for ln in feedback.split("\n")[:3] if not re.match(r"^Score\s+\d\s*:", ln.strip(), re.IGNORECASE)]
    out["reasoning"] = (" ".join(reasoning_lines)[:300]).strip() if reasoning_lines else feedback[:300]
    search_text = text
    label_to_key = [
        (r"novelty|originality", "novelty"),
        (r"usability|practicality", "usability"),
    ]
    for label_pat, out_key in label_to_key:
        if out[out_key] is not None:
            continue
        pat = r"(?:" + label_pat + r")\s*[:\=\(\s]+\s*([1-5])\b"
        km = re.search(pat, search_text, re.IGNORECASE)
        if km:
            out[out_key] = int(km.group(1))
    for key in ("novelty", "usability"):
        if out[key] is not None:
            continue
        pat_underscore = key.replace("_", r"\s*_") + r"\s*[:\(]?\s*([1-5])"
        pat_space = key.replace("_", r"\s+") + r"\s*[:\(]?\s*([1-5])"
        km = re.search(pat_underscore, search_text, re.IGNORECASE) or re.search(pat_space, search_text, re.IGNORECASE)
        if km:
            out[key] = int(km.group(1))
    # Overall = mean of novelty and usability
    if out["novelty"] is not None and out["usability"] is not None:
        out["overall_score"] = round((out["novelty"] + out["usability"]) / 2)
    else:
        m = re.search(r"\[RESULT\]\s*(\d+)", text, re.IGNORECASE)
        if m:
            out["overall_score"] = int(m.group(1))
    print(f"[parse_prometheus_response] -> novelty={out['novelty']}, usability={out['usability']}, overall={out['overall_score']}")
    return out

def call_judge(row, temperature=0.2) -> dict:
    """Call Prometheus judge; return dict with overall_score, solution_quality, novelty, usability, reasoning."""
    empty = {"overall_score": None, "solution_quality": None, "novelty": None, "usability": None, "reasoning": ""}
    if judge_model is None or judge_tokenizer is None:
        empty["reasoning"] = "Prometheus model not loaded"
        return empty
    instruction = build_prometheus_instruction(row)
    messages = [{"role": "system", "content": PROMETHEUS_SYSTEM}, {"role": "user", "content": instruction}]
    try:
        print(messages)
        text = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = judge_tokenizer([text], return_tensors="pt").to(judge_model.device)
        out_ids = judge_model.generate(**inputs, max_new_tokens=256, do_sample=(temperature > 0), temperature=temperature if temperature > 0 else 1e-5, pad_token_id=judge_tokenizer.eos_token_id)
        gen = out_ids[0][inputs["input_ids"].shape[1]:]
        response_text = judge_tokenizer.decode(gen, skip_special_tokens=True)
        print(response_text)
        return parse_prometheus_response(response_text)
    except Exception as e:
        empty["reasoning"] = str(e)[:200]
        return empty


def call_judge_batch(rows_list):
    """Run Prometheus on a batch of rows. rows_list = list of (idx, row) from judge_long."""
    if not rows_list or judge_model is None or judge_tokenizer is None:
        return [empty.copy() for _ in rows_list]
    print(f"[call_judge_batch] batch size={len(rows_list)}")
    batch_messages = [[{"role": "system", "content": PROMETHEUS_SYSTEM}, {"role": "user", "content": build_prometheus_instruction(row)}] for _, row in rows_list]
    texts = [judge_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) for msgs in batch_messages]
    if texts:
        print(f"[call_judge_batch] first prompt (last 500 chars of templated text):\n{repr(texts[0][-500:])}\n---")
    batch_responses = _run_judge_batch_raw(texts, max_new_tokens=256)
    if batch_responses:
        print(f"[call_judge_batch] first judge response (first 400 chars):\n{repr(batch_responses[0][:400])}\n---")
    return [parse_prometheus_response(t) for t in batch_responses]


def call_judge_aut_by_use_batch(judge_long_df):
    """Run judge in batches; map results back to one score per row.

    Aggregation of responses across uses:
    - json: One prompt per row (all uses in one message). One response per row.
      Parsed JSON gives overall.{novelty, usability, result}. We use those directly.
      If overall is missing, novelty/usability are averaged over uses[] and overall = round((n+u)/2).
    - per_use: One prompt per use (flattened). We split responses by row then aggregate:
      novelty = round(mean(per-use novelty)), usability = round(mean(per-use usability)),
      overall_score = round((novelty + usability) / 2). Only parsed uses contribute to the mean.
    """
    n_rows = len(judge_long_df)
    if judge_model is None or judge_tokenizer is None:
        return [dict(empty, reasoning="Judge not loaded") for _ in range(n_rows)]

    if JUDGE_AUT_BY_USE == "json":
        # One templated prompt per row; batch run then parse each (row order preserved)
        prompts = []
        for row_i, (idx, row) in enumerate(judge_long_df.iterrows()):
            obj = _extract_object_from_problem(row.get("problem_text", "") or "")
            reply = row.get("reply") or ""
            print(f"[Judge JSON] row_i={row_i} idx={idx} reply_len={len(reply)}")
            uses_list = split_reply_into_uses(reply)
            if not uses_list:
                print(f"[Judge JSON] row_i={row_i} -> no uses, appending None")
                prompts.append(None)  # placeholder, will return empty for this row
                continue
            inst = build_judge_json_instruction(obj, uses_list)
            print(f"[Judge JSON] row_i={row_i} object={obj!r} n_uses={len(uses_list)} instruction (first 400 chars):\n{inst[:400]}\n---")
            # Prometheus expects Mistral chat template (model card: "apply the conversation template of Mistral (not applying it might lead to unexpected behaviors)")
            messages = [{"role": "system", "content": PROMETHEUS_SYSTEM}, {"role": "user", "content": inst}]
            text = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            prompts.append(text)
        # Run one object at a time (each row = one object×method; no batching)
        valid_indices = [i for i, p in enumerate(prompts) if p is not None]
        valid_prompts = [prompts[i] for i in valid_indices]
        judge_json_batch_size = 1  # judge each object separately
        print(f"[Judge JSON] total rows={n_rows}, valid_prompts={len(valid_prompts)}, batch_size={judge_json_batch_size} (each object separately)")
        all_responses = []
        for start in tqdm(range(0, len(valid_prompts), judge_json_batch_size), desc="Judge (AUT by-use batch)"):
            batch = valid_prompts[start : start + judge_json_batch_size]
            batch_resp = _run_judge_batch_raw(batch, max_new_tokens=1024)
            if start == 0 and batch_resp:
                print(f"[Judge JSON] first batch first response (first 500 chars):\n{repr(batch_resp[0][:500])}\n---")
            all_responses.extend(batch_resp)
        # Map back to row order
        score_by_valid = [None] * len(valid_indices)
        for i, resp in enumerate(all_responses):
            if i < len(valid_indices):
                score_by_valid[i] = resp
        result = []
        j = 0
        for i in range(n_rows):
            if prompts[i] is None:
                result.append(dict(empty))
            else:
                raw = score_by_valid[j] if j < len(score_by_valid) else ""
                parsed = parse_judge_json_response(raw)
                if len(parsed.get("uses", [])) == 0:
                    print(f"[Judge JSON] row_i={i} -> 0 uses; judge raw response (first 600 chars):\n{repr(raw[:600])}\n---")
                j += 1
                obj = _extract_object_from_problem(judge_long_df.iloc[i].get("problem_text", "") or "")
                reason = (parsed.get("reasoning") or "").strip()
                if not reason:
                    reason = f"({len(parsed.get('uses', []))} uses scored)"
                reasoning = f"Object: {obj}. {reason}"
                result.append({
                    "overall_score": parsed.get("overall_score"),
                    "solution_quality": None,
                    "novelty": parsed.get("novelty"),
                    "usability": parsed.get("usability"),
                    "reasoning": reasoning,
                })
        return result

    if JUDGE_AUT_BY_USE == "per_use":
        # Flatten: one prompt per use across all rows; record (row_idx, num_prompts) to split back
        sys_msg = "You are a fair evaluator. Output only valid JSON in the requested format."
        all_prompts = []
        row_num_prompts = []  # (row_idx, num_prompts) in iterrows order
        for row_i, (idx, row) in enumerate(judge_long_df.iterrows()):
            obj = _extract_object_from_problem(row.get("problem_text", "") or "")
            reply = row.get("reply") or ""
            print(f"[Judge per_use] row_i={row_i} idx={idx} reply_len={len(reply)}")
            uses_list = split_reply_into_uses(reply)
            if not uses_list:
                print(f"[Judge per_use] row_i={row_i} -> no uses")
                row_num_prompts.append((idx, 0))
                continue
            if row_i == 0:
                inst0 = build_judge_per_use_instruction(obj, uses_list[0])
                print(f"[Judge per_use] row_i=0 first instruction (first 350 chars):\n{inst0[:350]}\n---")
            prompts = [
                judge_tokenizer.apply_chat_template(
                    [{"role": "system", "content": sys_msg}, {"role": "user", "content": build_judge_per_use_instruction(obj, use_text)}],
                    tokenize=False, add_generation_prompt=True,
                )
                for use_text in uses_list
            ]
            row_num_prompts.append((idx, len(prompts)))
            all_prompts.extend(prompts)
        print(f"[Judge per_use] total prompts={len(all_prompts)}, batch_size={JUDGE_BATCH_SIZE}")
        # Run in chunks
        all_responses = []
        for start in tqdm(range(0, len(all_prompts), JUDGE_BATCH_SIZE), desc="Judge (AUT by-use batch)"):
            batch = all_prompts[start : start + JUDGE_BATCH_SIZE]
            # 384 tokens so prose + [RESULT] N is not truncated
            batch_resp = _run_judge_batch_raw(batch, max_new_tokens=384)
            if start == 0 and batch_resp:
                print(f"[Judge per_use] first batch first response:\n{repr(batch_resp[0][:400])}\n---")
            all_responses.extend(batch_resp)
        # Split by row and aggregate
        pos = 0
        result = []
        for idx, n in row_num_prompts:
            if n == 0:
                result.append(dict(empty))
                continue
            slice_resp = all_responses[pos : pos + n]
            pos += n
            reasoning, novelties, usabilities, overall_scores = [], [], [], []
            for t in slice_resp:
                parsed = parse_judge_per_use_response(t)
                if parsed.get("novelty") is not None:
                    novelties.append(parsed["novelty"])
                if parsed.get("usability") is not None:
                    usabilities.append(parsed["usability"])
                if parsed.get("overall_score") is not None:
                    overall_scores.append(parsed["overall_score"])
                if parsed.get('reasoning') is not None:
                    reasoning.append(parsed['reasoning'])
            n_mean = round(sum(novelties) / len(novelties)) if novelties else None
            u_mean = round(sum(usabilities) / len(usabilities)) if usabilities else None
            if n_mean is not None and u_mean is not None:
                overall = round((n_mean + u_mean) / 2)
            elif overall_scores:
                overall = round(sum(overall_scores) / len(overall_scores))
            else:
                result.append(dict(empty))
                continue
            result.append({"overall_score": overall, "novelty": n_mean, "usability": u_mean, "reasoning": "\n".join(reasoning)})
        return result

    return [dict(empty) for _ in range(n_rows)]


In [26]:

# Load Prometheus 2  
judge_model = None
judge_tokenizer = None
if True:
    # Free GPU memory so the judge can load on GPU (otherwise a previously loaded Qwen 7B leaves no VRAM)
    import gc
    if "loader" in dir() and hasattr(loader, "_model") and loader._model is not None:
        del loader._model
        loader._model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory cleared. Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL, use_fast=True)
    judge_model = AutoModelForCausalLM.from_pretrained(
        JUDGE_MODEL, torch_dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True
    )
    judge_model.eval()
    _dev = next(judge_model.parameters()).device
    print(f"Loaded judge: {JUDGE_MODEL} (device: {_dev})")
    if _dev.type == "cpu":
        print("WARNING: Judge is on CPU — expect 2–5+ min per example.")

# Ensure judge_long exists (e.g. if Build judge table cell was skipped due to earlier error)
try:
    judge_long
except NameError:
    id_to_A = {item["id"]: item["A"] for item in eval_items}
    judge_long = combined_df[["id", "method", "reply"]].copy()
    judge_long = judge_long.rename(columns={"id": "problem_id"})
    judge_long["problem_text"] = judge_long["problem_id"].map(id_to_A)
    print("Built judge_long from combined_df (Build judge table cell was skipped).")

# Batched Prometheus inference (uses GPU better on powerful machines)
JUDGE_BATCH_SIZE = 64  # 24–32 on 40GB+ GPU; 1 to disable batching
empty = {"overall_score": None, "solution_quality": None, "novelty": None, "usability": None, "reasoning": ""}

# Run judge on all rows (all methods) together; one progress bar, batched over full judge_long
judge_scores = []

print(f"[Judge] TASK_TYPE={TASK_TYPE.value}, JUDGE_AUT_BY_USE={JUDGE_AUT_BY_USE!r}, JUDGE_BATCH_SIZE={JUDGE_BATCH_SIZE}, rows={len(judge_long)}")
if TASK_TYPE.value == "aut" and JUDGE_AUT_BY_USE:
    judge_scores = call_judge_aut_by_use_batch(judge_long)
elif judge_model is not None and JUDGE_BATCH_SIZE > 1:
    rows_list = list(judge_long.iterrows())
    for start in tqdm(range(0, len(rows_list), JUDGE_BATCH_SIZE), desc="Judge"):
        batch = rows_list[start : start + JUDGE_BATCH_SIZE]
        judge_scores.extend(call_judge_batch(batch))
else:
    for idx, row in tqdm(judge_long.iterrows(), total=len(judge_long), desc="Judge"):
        judge_scores.append(call_judge(row))

judge_long["judge_overall"] = [x.get("overall_score") for x in judge_scores]
judge_long["judge_novelty"] = [x.get("novelty") for x in judge_scores]
judge_long["judge_usability"] = [x.get("usability") for x in judge_scores]
judge_long["judge_reasoning"] = [x.get("reasoning") for x in judge_scores]
# List of parsed uses per reply (from split_reply_into_uses) for inspection
judge_long["reply_uses"] = [split_reply_into_uses(row.get("reply") or "") for _, row in judge_long.iterrows()]
judge_long.to_csv(EVAL_OUTPUT_DIR / "judge_table_long_with_scores.csv", index=False)
print("Saved: eval_outputs/judge_table_long_with_scores.csv")

NameError: name 'torch' is not defined

In [ ]:
pd.set_option('display.max_colwidth', None)
judge_long.iloc[:10][['judge_overall', 'judge_novelty', 'judge_usability', 'judge_reasoning']]
# judge_long.groupby('method').value_counts('judge_usability')  

In [20]:
"""Aggregate judge statistics by method and save.
'baseline' = SFT run (base model, minimal prompt → long prose). 'steering_baseline' = bridge run (format rules → short bullets). See note above for why scores differ."""
for col in ["judge_overall", "judge_novelty", "judge_usability"]:
    judge_long[col] = pd.to_numeric(judge_long[col], errors="coerce")
stats = judge_long.groupby("method").agg(
    mean_overall=("judge_overall", "mean"),
    mean_novelty=("judge_novelty", "mean"),
    mean_usability=("judge_usability", "mean"),
    n=("judge_overall", "count"),
).round(3)
stats_path = EVAL_OUTPUT_DIR / "judge_statistics_by_method.csv"
stats.to_csv(stats_path)
# Show table without all-NaN columns so output is readable (full stats still in CSV)
stats_display = stats.dropna(axis=1, how="all")
print("Judge statistics by method:")
print(stats_display)
print(f"\nSaved: {stats_path}")

Judge statistics by method:
                             mean_overall  mean_novelty  mean_usability   n
method                                                                     
baseline                              6.6           5.8             7.5  10
postsft                               6.8           6.2             7.6  10
steering_alpha_0.0                    5.2           3.8             6.6  10
steering_alpha_0.25                   5.0           3.2             6.8  10
steering_alpha_0.5                    5.5           3.5             7.2  10
steering_alpha_0.75                   5.2           3.8             6.6  10
steering_alpha_1.0                    4.9           3.8             6.3  10
steering_alpha_1.25                   5.3           4.3             6.4  10
steering_alpha_1.5                    4.8           4.5             5.3  10
steering_baseline                     5.3           3.6             6.8  10
steering_few_shot                     7.6           7.0     

In [18]:
judge_long

,problem_id,method,reply,problem_text,judge_overall,judge_novelty,judge_usability,judge_reasoning
0,AUT_016_v2,baseline,1. **Polishing Precious Metals**: Sandpaper ca...,Give 8 unconventional uses for sandpaper focus...,NaN,None,None,The response provided a variety of uses for sa...
1,AUT_018_v2,baseline,1. **Thermal Conductivity Tester**: Use the me...,Give 8 unconventional uses for a metal spoon f...,NaN,None,None,The response provides a variety of uses for a ...
2,AUT_022_v2,baseline,1. **Mechanical Fastener**: In mechanical desi...,Give 8 unconventional uses for a binder clip f...,4.0,None,None,The response provides a variety of uses for bi...
3,AUT_028_v2,baseline,1. **User Interface Wireframing:** Use sticky ...,Give 8 unconventional uses for sticky notes fo...,4.0,None,None,The response provided a variety of uses for st...
4,AUT_042_v2,baseline,1. **Shear Stress Reducer**: In the field of f...,Give 8 unconventional uses for a towel focusin...,3.0,None,None,The response provides a reasonable number of u...
...,...,...,...,...,...,...,...,...
125,AUT_022_v2_1,steering_few_shot,Mechanism Type: Flexible Containerization & Mo...,Give 8 unconventional uses for a cardboard box...,4.0,None,None,The response provided a variety of uses for a ...
126,AUT_057_v2,steering_few_shot,Mechanism Type: Reconfiguration \n\nRule: Trea...,Give 8 unconventional uses for a sticky note f...,3.0,None,None,The response provided a variety of unconventio...
127,AUT_065_v2,steering_few_shot,Mechanism Type: Re-representation\n\nRule: Tre...,Give 8 unconventional uses for a microfiber cl...,NaN,None,None,The response provides two clear uses for a mic...
128,AUT_104_v2,steering_few_shot,Mechanism type: Re-representation\nRule: Treat...,Give 8 unusual uses for a coffee filter focusi...,4.0,None,None,"The response provided a single, clear use of a..."


## Prometheus LLM-as-judge

Use Prometheus to score each (problem, method, reply). Scores and reasoning are parsed from the model output for aggregation.

In [ ]:
"""Wide table: one row per problem, one column per method (reply text)."""

wide_rows = []
for eval_idx, item in enumerate(eval_items):
    row = {
        "eval_idx": eval_idx,
        "problem_id": item["id"],
        "problem_text": item["A"],
        "gold_B": item["B"]["rule"],
        "gold_C": item["C"] if isinstance(item["C"], str) else " | ".join(item["C"]),
    }
    for method in combined_df["method"].unique():
        r = combined_df[(combined_df["eval_idx"] == eval_idx) & (combined_df["method"] == method)].iloc[0]
        row[f"reply_{method}"] = r["reply"]
    wide_rows.append(row)

judge_wide = pd.DataFrame(wide_rows)
judge_wide_path = EVAL_OUTPUT_DIR / "judge_table_wide.csv"
judge_wide.to_csv(judge_wide_path, index=False)
print(f"Judge table (wide): {judge_wide_path}")
display(judge_wide[["problem_id", "problem_text"] + [c for c in judge_wide.columns if c.startswith("reply_")]].head(2))

In [22]:
"""Export markdown for pasting into LLM judge: one section per problem with all method replies."""

md_lines = ["# Eval: Problem × Method → Reply (for LLM judge)", ""]
for eval_idx, item in enumerate(eval_items):
    md_lines.append(f"## Problem {eval_idx + 1}: {item['id']}")
    md_lines.append("") 
    md_lines.append("**Problem:** " + item["A"].replace("\n", " ")[:500])
    md_lines.append("")
    md_lines.append("**Gold mechanism (B):** " + item["B"]["rule"])
    md_lines.append("")
    gold_c = item["C"] if isinstance(item["C"], str) else " | ".join(item["C"])
    md_lines.append("**Gold solution (C):** " + gold_c[:400])
    md_lines.append("")
    for method in combined_df["method"].unique():
        r = combined_df[(combined_df["eval_idx"] == eval_idx) & (combined_df["method"] == method)].iloc[0]
        md_lines.append(f"### Reply [{method}]")
        md_lines.append("")
        md_lines.append(r["reply"].strip())
        md_lines.append("")
    md_lines.append("---")
    md_lines.append("")

judge_md_path = EVAL_OUTPUT_DIR / "judge_prompt.md"
judge_md_path.write_text("\n".join(md_lines), encoding="utf-8")
print(f"Judge markdown: {judge_md_path}")
print("First 2000 chars:")
print("\n".join(md_lines)[:2000])

Judge markdown: eval_outputs/judge_prompt.md
First 2000 chars:
# Eval: Problem × Method → Reply (for LLM judge)

## Problem 1: AUT_016_v2

**Problem:** Give 8 unconventional uses for sandpaper focusing on precision work (non-destructive options included).

**Gold mechanism (B):** Treat it as a controllable abrasive transfer function (grit as a tuning knob).

**Gold solution (C):** Friction pad to prevent a clamp from slipping without overtightening | Micro-trimming a plastic shim to tune a fit by tiny increments | Surface roughness standard: compare tactile roughness across samples | Edge deburr on 3D prints using a single light pass per side | Grip enhancer on a screwdriver handle (wrap strip temporarily) | Controlled dulling of a glossy patch to reduce glare in a vision t

### Reply [baseline]

1. **Polishing Precious Metals**: Sandpaper can be used to polish metals like gold, silver, and copper without causing damage. By using finer grits of sandpaper, you can achieve a smooth, poli

In [10]:
import json
import re

def parse_judge_per_use_json(text: str) -> dict:
    """
    Expect JSON exactly:
    {
      "analysis": {"functional_property": "...", "reasoning": "..."},
      "scores": {"novelty": int 1..10, "usability": int 1..10}
    }
    """
    out = {"novelty": None, "usability": None, "overall_score": None, "reasoning": ""}
    raw = (text or "").strip()

    # Strip accidental wrappers
    if "[/INST]" in raw:
        raw = raw.split("[/INST]", 1)[-1].strip()

    # If model returned extra text, extract the first {...} block
    if not raw.startswith("{"):
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            raw = raw[start:end+1]

    try:
        d = json.loads(raw)
        n = int(d["scores"]["novelty"])
        u = int(d["scores"]["usability"])
        if not (1 <= n <= 10 and 1 <= u <= 10):
            raise ValueError(f"Scores out of range: n={n}, u={u}")
        out["novelty"] = n
        out["usability"] = u
        out["overall_score"] = round((n + u) / 2)
        out["reasoning"] = (d.get("analysis", {}).get("reasoning") or "")[:200]
        return out
    except Exception:
        # Fallback: if JSON parse fails, return empty (you can log raw for debugging)
        out["reasoning"] = raw[:200]
        return out

In [11]:
from pydantic import BaseModel, Field

class PerUseAnalysis(BaseModel):
    functional_property: str = Field(description="Which object property is used (e.g., friction, stiffness).")
    reasoning: str = Field(description="<= 20 words, justification for scores.")

class PerUseScores(BaseModel):
    novelty: int = Field(ge=1, le=10)
    usability: int = Field(ge=1, le=10)

class PerUseJudgeOut(BaseModel):
    analysis: PerUseAnalysis
    scores: PerUseScores

In [27]:
import asyncio
from openai import AsyncOpenAI

VLLM_BASE_URL = "http://localhost:8000/v1"  # change if remote
JUDGE_MODEL_VLLM = "prometheus-eval/prometheus-7b-v2.0"
JUDGE_MODEL_VLLM = "Qwen/Qwen3-32B"
MAX_CONCURRENT_JUDGE = 32
MAX_RETRIES = 3
TIMEOUT_S = 45.0

vllm_client = AsyncOpenAI(base_url=VLLM_BASE_URL, api_key="vllm-runs-locally")
judge_sema = asyncio.Semaphore(MAX_CONCURRENT_JUDGE)

async def _judge_one_per_use(object_name: str, use_text: str, *, temperature: float = 0.0) -> dict:
    """
    Returns parsed dict: {novelty, usability, overall_score, reasoning}
    """
    prompt = build_judge_per_use_instruction(object_name, use_text)

    for attempt in range(MAX_RETRIES):
        async with judge_sema:
            try:
                resp = await asyncio.wait_for(
                    vllm_client.chat.completions.create(
                        model=JUDGE_MODEL_VLLM,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=256,
                        extra_body={
                            "structured_outputs": {
                                "json": PerUseJudgeOut.model_json_schema()
                            }
                        },
                    ),
                    timeout=TIMEOUT_S,
                )
                text = (resp.choices[0].message.content or "").strip()
                return parse_judge_per_use_json(text)

            except asyncio.TimeoutError:
                if attempt == MAX_RETRIES - 1:
                    return {"novelty": None, "usability": None, "overall_score": None, "reasoning": "timeout"}
            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    return {"novelty": None, "usability": None, "overall_score": None, "reasoning": str(e)[:200]}

    return {"novelty": None, "usability": None, "overall_score": None, "reasoning": "failed"}

In [46]:
MAX_CONCURRENT_JUDGE = 32
sema = asyncio.Semaphore(MAX_CONCURRENT_JUDGE)

async def _judge_one(prompt_text: str, temperature: float = 0.0, timeout_s: float = 60.0) -> str:
    async with sema:
        resp = await asyncio.wait_for(
            vllm_client.chat.completions.create(
                model=JUDGE_MODEL_VLLM,
                messages=[
                    {"role": "system", "content": PROMETHEUS_SYSTEM},
                    {"role": "user", "content": prompt_text},
                ],
                temperature=temperature,
                max_tokens=1024,
            ),
            timeout=timeout_s,
        )
        return resp.choices[0].message.content or ""

async def judge_rows_vllm(judge_long_df, temperature: float = 0.0):
    prompts = [build_prometheus_instruction(row) for _, row in judge_long_df.iterrows()]
    tasks = [asyncio.create_task(_judge_one(p, temperature=temperature)) for p in prompts]
    texts = await asyncio.gather(*tasks, return_exceptions=True)

    out = []
    for i, t in enumerate(texts):
        if isinstance(t, Exception):
            out.append({"input_prompt": prompts[i],
                "overall_score": None, "novelty": None, "usability": None, "reasoning": str(t)})
        else:
            cur = parse_prometheus_response(t)
            out.append({**cur, "input_prompt": prompts[i], "output_response": str(t)})
    return out

In [47]:
# Judge all rows via vLLM
judge_scores = await judge_rows_vllm(judge_long, temperature=0.0)

judge_long["judge_overall"] = [x.get("overall_score") for x in judge_scores]
judge_long["judge_novelty"]  = [x.get("novelty") for x in judge_scores]
judge_long["judge_usability"]= [x.get("usability") for x in judge_scores]
judge_long["judge_reasoning"]= [x.get("reasoning") for x in judge_scores]
judge_scores

[build_prometheus_instruction] reply len=1663 preview: '1. **Polishing Precious Metals**: Sandpaper can be used to polish metals like gold, silver, and copper without causing damage. By using finer grits of sandpaper, you can achieve a smooth, polished sur'
[build_prometheus_instruction] reply len=2001 preview: '1. **Thermal Conductivity Tester**: Use the metal spoon to test the thermal conductivity of different surfaces or materials. By placing the spoon against various objects, you can gauge how quickly the'
[build_prometheus_instruction] reply len=2027 preview: '1. **Mechanical Fastener**: In mechanical design, binder clips can be used as temporary fasteners to hold parts together during assembly or disassembly processes. They can secure components in place w'
[build_prometheus_instruction] reply len=2067 preview: '1. **User Interface Wireframing:** Use sticky notes to quickly sketch out the layout of a website or app interface. This can help in brainstorming different design ideas,

[parse_prometheus_response] input len=2227, first 250 chars: '<think>\nI need to score the response to the instruction "Give 8 unconventional uses for sandpaper focusing on precision work (non-destructive options included)." The response provided only 5 points, and the fifth point is incomplete.\n\nLet me analyze '
[parse_prometheus_response] -> novelty=1, usability=1, overall=1
[parse_prometheus_response] input len=2115, first 250 chars: '<think>\nI need to score the response to the instruction "Give 8 unconventional uses for a metal spoon focusing on heat transfer and sensing." The response provided only 4 uses, and the fourth one is incomplete.\n\nLet me analyze the response based on t'
[parse_prometheus_response] -> novelty=3, usability=3, overall=3
[parse_prometheus_response] input len=2382, first 250 chars: "<think>\nI need to score the given response on two dimensions: novelty and usability, each on a scale of 1-5. Then I'll provide a one-sentence reasoning and calculate the rou

[{'overall_score': 1,
  'solution_quality': None,
  'novelty': 1,
  'usability': 1,
  'reasoning': '<think> I need to score the response to the instruction "Give 8 unconventional uses for sandpaper focusing on precision work (non-destructive options included)." The response provided only 5 points, and the fifth point is incomplete.',
  'input_prompt': '###Task Description:\nScore the following response on its own merits. No reference answer. 1. In the FIRST line state exactly: novelty (N), usability (N) with N 1-5 each. Example: novelty (4), usability (3). 2. One sentence reasoning. 3. End with [RESULT] followed by one integer 1-5 (the rounded mean of novelty and usability).\n\n###Instruction given to the model:\nGive 8 unconventional uses for sandpaper focusing on precision work (non-destructive options included).\n\n###Response to score:\n1. **Polishing Precious Metals**: Sandpaper can be used to polish metals like gold, silver, and copper without causing damage. By using finer grits

In [ ]:
judge_scores = asyncio.run(judge_rows_vllm(judge_long, temperature=0.0))

In [ ]:
"""Aggregate judge statistics by method and save.
'baseline' = SFT run (base model, minimal prompt → long prose). 'steering_baseline' = bridge run (format rules → short bullets). See note above for why scores differ."""
for col in ["judge_overall", "judge_novelty", "judge_usability"]:
    judge_long[col] = pd.to_numeric(judge_long[col], errors="coerce")
stats = judge_long.groupby("method").agg(
    mean_overall=("judge_overall", "mean"),
    mean_novelty=("judge_novelty", "mean"),
    mean_usability=("judge_usability", "mean"),
    n=("judge_overall", "count"),
).round(3)
stats_path = EVAL_OUTPUT_DIR / "judge_statistics_by_method.csv"
stats.to_csv(stats_path)
# Show table without all-NaN columns so output is readable (full stats still in CSV)
stats_display = stats.dropna(axis=1, how="all")
print("Judge statistics by method:")
print(stats_display)
print(f"\nSaved: {stats_path}")

Judge statistics by method:
                             mean_overall  mean_novelty  mean_usability   n
method                                                                     
baseline                              2.2           2.3             2.3  10
postsft                               2.5           2.5             2.8  10
steering_alpha_0.0                    1.9           1.8             2.0  10
steering_alpha_0.25                   1.6           1.7             1.7  10
steering_alpha_0.5                    2.1           2.1             2.3  10
steering_alpha_0.75                   2.2           2.2             2.4  10
steering_alpha_1.0                    2.5           2.4             2.7  10
steering_alpha_1.25                   2.3           2.4             2.5  10
steering_alpha_1.5                    2.0           2.2             2.0  10
steering_baseline                     1.8           1.7             2.0  10
steering_few_shot                     2.4           2.4     

: 

In [ ]:

judge_scores

[{'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'Connection error.'},
 {'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'Connection error.'},
 {'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'Connection error.'},
 {'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'Connection error.'},
 {'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'Connection error.'},
 {'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'Connection error.'},
 {'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'Connection error.'},
 {'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'Connection error.'},
 {'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'Connection error.'},
 {'overall_score': None,
  'novelty': None,
  'usability': None,
  'reasoning': 'C